# Week 12 Puzzles — Open Data APIs & Data Engineering

> **NS5116 Computational Neuroscience — Spring 2026**

This notebook is divided into two parts:

- **Part 1 — Guided Practice (Puzzles 1–10):** Each puzzle includes a worked solution.
- **Part 2 — Independent Practice (Puzzles 11–20):** Write your own solution in the empty code cells.

> **Note:** Puzzles that call external APIs use mocked responses so the notebook runs without network access. The patterns are identical to real API calls.

---

## Part 1 — Guided Practice (with solutions)

### Puzzle 1 — Parse a JSON API Response into a DataFrame

Given a mock JSON API response dict, extract the `records` list and convert it into a pandas DataFrame.

Print the shape and first 3 rows.

In [ ]:
import pandas as pd

# Mock API response
api_response = {
    "status": "ok",
    "total": 5,
    "records": [
        {"station": "Songshan",  "county": "Taipei",    "pm25": "22", "aqi": "65",  "date": "2024-01-15"},
        {"station": "Banqiao",   "county": "Taipei",    "pm25": "25", "aqi": "70",  "date": "2024-01-15"},
        {"station": "Zuoying",   "county": "Kaohsiung", "pm25": "35", "aqi": "90",  "date": "2024-01-15"},
        {"station": "Xiaogang",  "county": "Kaohsiung", "pm25": "40", "aqi": "105", "date": "2024-01-15"},
        {"station": "Xitun",     "county": "Taichung",  "pm25": "28", "aqi": "75",  "date": "2024-01-15"},
    ],
}

df = pd.DataFrame(api_response["records"])

print(f"Shape: {df.shape}")
print(f"\n{df.head(3).to_string(index=False)}")

### Puzzle 2 — Fix Column Types with `pd.to_numeric`

API responses often return numbers as strings. Convert `pm25` and `aqi` from strings to numeric types using `pd.to_numeric(errors='coerce')`.

Print dtypes before and after.

In [ ]:
print("Before:")
print(df.dtypes)
print()

df["pm25"] = pd.to_numeric(df["pm25"], errors="coerce")
df["aqi"]  = pd.to_numeric(df["aqi"],  errors="coerce")

print("After:")
print(df.dtypes)

### Puzzle 3 — Parse Date Strings with `pd.to_datetime`

Convert the `date` column from string to datetime. Then extract the month and day-of-week as new columns.

Print the updated DataFrame.

In [ ]:
df["date"] = pd.to_datetime(df["date"], format="%Y-%m-%d", errors="coerce")

df["month"]     = df["date"].dt.month
df["day_of_week"] = df["date"].dt.day_name()

print(df[["station", "date", "month", "day_of_week"]].to_string(index=False))

### Puzzle 4 — Handle Missing Values with `dropna` and `fillna`

Create a DataFrame with deliberate missing values. Demonstrate:
1. `dropna(subset=[...])` to remove rows missing critical columns
2. `fillna(value)` to fill missing categorical values
3. Print the row count before and after each operation

In [ ]:
import numpy as np

df_dirty = pd.DataFrame({
    "station": ["A", "B", None, "D", "E"],
    "pm25":    [22,  np.nan, 35,  28,  np.nan],
    "county":  ["Taipei", "Taipei", "Kaohsiung", None, "Taichung"],
})

print(f"Original: {len(df_dirty)} rows")
print(df_dirty)
print()

# Drop rows where pm25 is missing (critical column)
df_clean = df_dirty.dropna(subset=["pm25"])
print(f"After dropna(pm25): {len(df_clean)} rows")

# Fill missing county with 'Unknown'
df_clean = df_clean.copy()
df_clean["county"] = df_clean["county"].fillna("Unknown")
print(f"After fillna(county): {len(df_clean)} rows")
print(df_clean)

### Puzzle 5 — Construct a URL with Query Parameters

Write a function `build_api_url(base_url, params)` that constructs a full URL with query parameters.

Example: `build_api_url("https://api.example.com/data", {"city": "Taipei", "limit": 100})`  
→ `"https://api.example.com/data?city=Taipei&limit=100"`

In [ ]:
from urllib.parse import urlencode

def build_api_url(base_url, params):
    """Construct a URL with query parameters.

    Args:
        base_url (str): Base endpoint URL.
        params (dict): Query parameters.

    Returns:
        str: Full URL with encoded parameters.
    """
    return f"{base_url}?{urlencode(params)}"


url = build_api_url(
    "https://data.epa.gov.tw/api/v2/aqx_p_432",
    {"api_key": "demo-key", "limit": 1000, "format": "json"},
)
print(url)

### Puzzle 6 — Handle HTTP Status Codes

Write a function `check_response(status_code)` that returns a human-readable message for common HTTP status codes.

Handle: 200 (OK), 400 (bad request), 401 (unauthorized), 404 (not found), 429 (rate limited), 500 (server error).

In [ ]:
def check_response(status_code):
    """Return a human-readable message for an HTTP status code."""
    messages = {
        200: "✓ OK — data received successfully",
        400: "✗ Bad Request — check query parameters",
        401: "✗ Unauthorized — check your API key",
        404: "✗ Not Found — endpoint does not exist",
        429: "✗ Rate Limited — wait before retrying",
        500: "✗ Server Error — try again later",
    }
    return messages.get(status_code, f"? Unknown status code: {status_code}")


for code in [200, 401, 404, 429, 500, 418]:
    print(f"  {code}: {check_response(code)}")

### Puzzle 7 — Merge Two DataFrames on a Common Key

Merge air quality readings with station metadata using `pd.merge()` on `station_id`.

Show both inner and left joins and explain the difference.

In [ ]:
readings = pd.DataFrame({
    "station_id": ["S01", "S02", "S03", "S04"],
    "pm25": [22, 35, 28, 40],
})

stations = pd.DataFrame({
    "station_id": ["S01", "S02", "S03"],
    "station_name": ["Songshan", "Zuoying", "Xitun"],
    "county": ["Taipei", "Kaohsiung", "Taichung"],
})

# Inner join — only matching rows
inner = pd.merge(readings, stations, on="station_id", how="inner")
print(f"Inner join ({len(inner)} rows):")
print(inner.to_string(index=False))

print()

# Left join — keep all readings, fill missing station info with NaN
left = pd.merge(readings, stations, on="station_id", how="left")
print(f"Left join ({len(left)} rows):")
print(left.to_string(index=False))

### Puzzle 8 — Cache API Data Locally

Write a function `fetch_or_load(cache_file, fetch_func)` that:
- Returns data from `cache_file` if it exists
- Otherwise calls `fetch_func()`, saves the result to `cache_file`, and returns it

Demonstrate with a mock fetch function.

In [ ]:
import os
import json

def fetch_or_load(cache_file, fetch_func):
    """Load from cache if available, otherwise fetch and cache."""
    if os.path.exists(cache_file):
        print(f"  Loading from cache: {cache_file}")
        with open(cache_file, encoding="utf-8") as f:
            return json.load(f)

    print(f"  Fetching fresh data...")
    data = fetch_func()

    os.makedirs(os.path.dirname(cache_file) or ".", exist_ok=True)
    with open(cache_file, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    return data


# Mock fetch function
def mock_fetch():
    return [{"station": "Songshan", "pm25": 22}, {"station": "Zuoying", "pm25": 35}]


# First call — fetches
data1 = fetch_or_load("aq_cache.json", mock_fetch)
# Second call — uses cache
data2 = fetch_or_load("aq_cache.json", mock_fetch)

print(f"\nData: {data1}")

os.remove("aq_cache.json")

### Puzzle 9 — Validate and Clean a Real-World Column

Real API data often has messy values. Clean a `pm25` column that contains:
- Valid numbers: `"22"`, `"35"`
- Missing markers: `""`, `"-"`, `"N/A"`
- Invalid text: `"error"`

Convert all to numeric, replace invalids with NaN, then report the count of valid vs. invalid values.

In [ ]:
df_messy = pd.DataFrame({
    "station": ["A", "B", "C", "D", "E", "F", "G"],
    "pm25_raw": ["22", "35", "", "-", "N/A", "error", "28"],
})

# Replace known missing markers with NaN before conversion
df_messy["pm25_raw"] = df_messy["pm25_raw"].replace(["", "-", "N/A"], pd.NA)

# Convert to numeric — remaining bad values become NaN
df_messy["pm25"] = pd.to_numeric(df_messy["pm25_raw"], errors="coerce")

valid   = df_messy["pm25"].notna().sum()
invalid = df_messy["pm25"].isna().sum()

print(f"Valid: {valid}, Invalid/Missing: {invalid}")
print()
print(df_messy.to_string(index=False))

### Puzzle 10 — Filter by Physical Plausibility

After type conversion, apply domain-knowledge filters: PM2.5 must be between 0 and 500 µg/m³.

Remove implausible values and report how many rows were dropped.

In [ ]:
df_range = pd.DataFrame({
    "station": ["A", "B", "C", "D", "E"],
    "pm25": [22, -5, 35, 800, 28],
})

before = len(df_range)
df_range = df_range[(df_range["pm25"] >= 0) & (df_range["pm25"] <= 500)]
after = len(df_range)

print(f"Before: {before} rows")
print(f"After:  {after} rows")
print(f"Dropped: {before - after} implausible values")
print()
print(df_range.to_string(index=False))

---

## Part 2 — Independent Practice (write your own solutions)

### Puzzle 11 — Parse a Nested JSON Response

Some APIs return nested JSON. Given:
```python
response = {"result": {"records": [...], "total": 100, "offset": 0}}
```
Write code to safely extract `records` and `total`, handling the case where `result` or `records` might be missing.

In [ ]:
# Your solution here


### Puzzle 12 — Convert API Column Names to Snake Case

API responses often use inconsistent casing. Write a function `to_snake_case(name)` and apply it to all column names in a DataFrame.

Test with columns: `["StationName", "PM2.5", "AQI Value", "date-recorded"]`

In [ ]:
# Your solution here


### Puzzle 13 — Paginate Through an API

Many APIs return results in pages. Simulate pagination by writing a function that:
1. Fetches page 1 (offset=0, limit=3)
2. Checks if there are more pages
3. Fetches page 2 (offset=3, limit=3)
4. Combines all records

Use a mock function instead of a real API.

In [ ]:
# Your solution here


### Puzzle 14 — Forward-Fill Missing Time Series Values

Create a time series DataFrame with some missing PM2.5 values. Use `fillna(method='ffill')` to carry the last valid value forward.

Print before and after to see the effect.

In [ ]:
# Your solution here


### Puzzle 15 — Merge Three DataFrames

You have three DataFrames: `readings` (station_id, pm25), `stations` (station_id, name, county), and `counties` (county, population).

Merge all three to get a single DataFrame with readings, station names, and county population.

In [ ]:
# Your solution here


### Puzzle 16 — Write a Complete Data Cleaning Pipeline Function

Write `clean_aq_data(df)` that applies all cleaning steps in order:
1. Convert `pm25` and `aqi` to numeric
2. Parse `date` to datetime
3. Drop rows with missing `pm25` or `date`
4. Filter to plausible PM2.5 range (0–500)
5. Fill missing `station` with `"Unknown"`

Test with a deliberately messy DataFrame.

In [ ]:
# Your solution here


### Puzzle 17 — Create a Data Quality Report

Write a function `data_quality_report(df)` that prints:
- Total rows and columns
- Missing value count per column
- Data types per column
- Number of duplicate rows

Test on a DataFrame with known quality issues.

In [ ]:
# Your solution here


### Puzzle 18 — Compute Per-Station Daily Averages

Given hourly PM2.5 readings, compute the daily average per station using `groupby(["station", "date"])`.

Create mock hourly data for 2 stations over 2 days.

In [ ]:
# Your solution here


### Puzzle 19 — Implement Exponential Backoff for Retries

Write a function `retry_with_backoff(func, max_retries=3)` that:
- Calls `func()`
- If it raises an exception, waits `2^attempt` seconds and retries
- After `max_retries` failures, raises the last exception

Test with a mock function that fails twice then succeeds.

In [ ]:
# Your solution here


### Puzzle 20 — Full API → Clean → Merge Pipeline *(Bonus)*

Combine skills from this week:
1. Parse a mock API response into a DataFrame
2. Fix all column types
3. Clean missing and implausible values
4. Merge with a station metadata DataFrame
5. Cache the cleaned result to a JSON file
6. Print a data quality report
7. Clean up the cache file

In [ ]:
# Your solution here
